# Project 1 ANSC 4040 Data Inspection

This notebook quickly inspects the project dataset and prints useful information such as file type, column names, shape, and summary statistics.

In [13]:
#%pip install pandas

In [3]:
#import pandas as pd
import pandas as pd

In [4]:
# Add file path and load the dataset
path = 'Data_set_prep_assignment_1.csv'
data = pd.read_csv(path)
data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,NaN,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,NaN,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


# Data Preprocessing

In [15]:
#Install Packages
%pip install pandas numpy scikit-learn ydata-profiling

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 11.1 MB 4.5 MB/s eta 0:00:01
     |████████████████████████████████| 399 kB 16.1 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 35.9 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 57.3 MB/s eta 0:00:01
     |████████████████████████████████| 80 kB 22.2 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 10.6 MB/s eta 0:00:01
     |████████████████████████████████| 134 kB 48.3 MB/s eta 0:00:01
     |████████████████████████████████| 2.7 MB 29.3 MB/s eta 0:00:01
     |████████████████████████████████| 2.8 MB 24.4 MB/s eta 0:00:01
     |████████████████████████████████| 174 kB 50.5 MB/s eta 0:00:01
     |████████████████████████████████| 472 kB 41.9 MB/s eta 0:00:01
     |████████████████████████████████| 664 kB 39.0 MB/s eta 0:00:01
     |████████████████████████████████| 10.0 MB 37.2 MB/s eta 0:00:01
     |███████████████████

In [16]:
#Import relevant packages
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from ydata_profiling import ProfileReport

/Users/katrinapayton/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.
/Users/katrinapayton/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [19]:
#Generate data profile report
profile = ProfileReport(data, title="Data Profile Report")
profile.to_file("data_profile_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 87.47it/s]


## Data Cleaning Plan
1. remove duplicates
2. perform Z score inspection
3. remove outliers
4. assess zero values
5. assess remaining missing values

In [6]:
#Drop Duplicates
data = data.drop_duplicates()

# Z Score Inspection

In [7]:
# Z score inspection for continuous measurement variables
measurement_columns = [
    "LactationNumber",
    "DaysInMilk",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
]

z_score_summary = []
for column in measurement_columns:
    values = data[column].dropna()
    mean = values.mean()
    standard_deviation = values.std(ddof=1)
    z_scores = (values - mean) / standard_deviation

    z_score_summary.append({
        "variable": column,
        "n": values.size,
        "mean": mean,
        "standard_deviation": standard_deviation,
        "missing": data[column].isna().sum(),
        "z_score_abs_greater_than_3": (z_scores.abs() > 3).sum(),
        "percent_z_score_abs_greater_than_3": (z_scores.abs() > 3).mean() * 100,
    })

z_score_summary = pd.DataFrame(z_score_summary).set_index("variable")
z_score_summary.round(3)

,n,mean,standard_deviation,missing,z_score_abs_greater_than_3,percent_z_score_abs_greater_than_3
variable,,,,,,
LactationNumber,6737898,2.294,1.388,1696870,51936,0.771
DaysInMilk,6737894,162.938,105.516,1696874,44123,0.655
Avgmilkflow,8434596,3.526,0.729,172,3803,0.045
Flow30_60Session,8434768,3.751,1.486,0,840,0.010
YieldFirst2Min_Session,8434768,7.491,2.014,0,0,0.000
YieldSession,8434768,13.928,3.646,0,20800,0.247
DurationSession_sec,8434768,236.855,54.839,0,31990,0.379


# Remove outliers

In [9]:
# Drop rows with an extreme Z-score in any measurement variable.
z_score_threshold = 3
outlier_mask = pd.Series(False, index=data.index)

for column in measurement_columns:
    values = data[column]
    mean = values.mean()
    standard_deviation = values.std(ddof=1)
    z_scores = (values - mean) / standard_deviation
    outlier_mask = outlier_mask | z_scores.abs().gt(z_score_threshold).fillna(False)

rows_before_outlier_removal = len(data)
data = data.loc[~outlier_mask].copy()
rows_removed = rows_before_outlier_removal - len(data)

print(f"Rows removed as outliers: {rows_removed}")
print(f"Rows remaining: {len(data)}")

Rows removed as outliers: 149391
Rows remaining: 8285377


# Assess Remaining Zero Values 

In [11]:
# Assess zero values in the dataset
zero_value_summary = []
for column in measurement_columns:
    zero_count = (data[column] == 0).sum()
    zero_percentage = (zero_count / len(data)) * 100
    zero_value_summary.append({
        "variable": column,
        "zero_count": zero_count,
        "zero_percentage": zero_percentage,
    })

zero_value_summary = pd.DataFrame(zero_value_summary).set_index("variable")
zero_value_summary.round(3)

,zero_count,zero_percentage
variable,,
LactationNumber,0,0.000
DaysInMilk,0,0.000
Avgmilkflow,0,0.000
Flow30_60Session,3098,0.037
YieldFirst2Min_Session,0,0.000
YieldSession,0,0.000
DurationSession_sec,0,0.000


In [ ]:
# Compare missing zero-values and nonzero values.
zero_missingness_summary = []
for column in measurement_columns:
    zero_rows = data[column].eq(0)
    nonzero_rows = data[column].ne(0) & data[column].notna()
    other_columns = data.columns.difference([column])

    zero_missing_any = data.loc[zero_rows, other_columns].isna().any(axis=1)
    nonzero_missing_any = data.loc[nonzero_rows, other_columns].isna().any(axis=1)

    zero_missingness_summary.append({
        "variable": column,
        "zero_rows": zero_rows.sum(),
        "zero_rows_with_other_missing": zero_missing_any.sum(),
        "zero_rows_with_other_missing_percent": zero_missing_any.mean() * 100 if len(zero_missing_any) else 0,
        "nonzero_rows_with_other_missing_percent": nonzero_missing_any.mean() * 100 if len(nonzero_missing_any) else 0,
    })

zero_missingness_summary = pd.DataFrame(zero_missingness_summary).set_index("variable")
zero_missingness_summary.round(3)

,zero_rows,zero_rows_with_other_missing,zero_rows_with_other_missing_percent,nonzero_rows_with_other_missing_percent
variable,,,,
LactationNumber,0,0,0.000,0.000
DaysInMilk,0,0,0.000,0.000
Avgmilkflow,0,0,0.000,20.347
Flow30_60Session,3098,604,19.496,20.347
YieldFirst2Min_Session,0,0,0.000,20.347
YieldSession,0,0,0.000,20.347
DurationSession_sec,0,0,0.000,20.347


The only zero values remianing are from the Flow30_60Session. These zero values are plausible, so I will keep them in my cleaned data set. 

# Assess Remaining Missing Values 

In [15]:
# Compare missing values across variables.
missing_value_summary = pd.DataFrame({
    "missing_count": data.isna().sum(),
    "missing_proportion": data.isna().mean(),
    "missing_percent": data.isna().mean() * 100,
    "nonmissing_count": data.notna().sum(),
})

missing_value_summary = missing_value_summary.sort_values(
    "missing_proportion", ascending=False
)
missing_value_summary.round({"missing_proportion": 4, "missing_percent": 2})

,missing_count,missing_proportion,missing_percent,nonmissing_count
DaysInMilk,1685833,0.2035,20.35,6599544
AnimalId,1685829,0.2035,20.35,6599548
LactationNumber,1685829,0.2035,20.35,6599548
ReproductionStatus,1685829,0.2035,20.35,6599548
Avgmilkflow,6,0.0000,0.00,8285371
EventDate,0,0.0000,0.00,8285377
Flow30_60Session,0,0.0000,0.00,8285377
YieldFirst2Min_Session,0,0.0000,0.00,8285377
YieldSession,0,0.0000,0.00,8285377
DurationSession_sec,0,0.0000,0.00,8285377


All missing values are for outcome variables excpet for 6 missing Avgmilkflow values. I will drop these rows because it is a very small percent of the data set. 

In [17]:
#drop rows with missing values for Avgmilkflow
data = data.dropna(subset=["Avgmilkflow"])